# YuE2 Studio 5.0.0 — Kaggle Studio (Cloudflare)

**This launches the actual upstream YuE2 Studio 5.0.0 interface and backend. It is not ComfyUI or Gradio.**

## Before running
1. Run **YuE2 Studio 5.0.0 — Assets Builder** once and save its output, or upload the resulting folder as a private Kaggle Dataset.
2. Attach that builder output here using **Add Input → Notebook Output Files** (or **Add Input → Your Datasets**).
3. Set Kaggle **Internet: ON** and **Accelerator: GPU T4 ×2**.
4. Run the single setup/launch cell below. It installs the Linux desktop runtime dependencies, starts the official Studio 5.0.0 AppImage in headless mode under a virtual display, checks that its local API is alive, clones the matching v5.0.0 frontend source, and launches the actual React/Vite Studio interface.
5. The cell prints a Cloudflare URL and a generated login. Open the URL in your browser and enter the login when asked.

## What is running
- **Backend:** the official YuE2 Studio 5.0.0 Linux AppImage, started with the upstream `--headless` option. Its own API listens locally on port 8791.
- **Interface:** the real React application from the matching upstream `v5.0.0` source tag, served by its documented Vite development server on port 3791. Its existing Vite proxy forwards Studio API routes to the local backend.
- **Remote access:** cloudflared Quick Tunnel exposes only the Vite interface. A generated HTTP Basic password protects the public tunnel; the Studio API port itself is not exposed directly.

The Studio will download its own selected engine/model components through its Models/Setup screens. Start with YuE2, then add ACE-Step or MiniMax components from the Studio when needed. Model downloads and generated songs are stored under `/kaggle/working/.local/share/yue2-studio`.

## Browser notes
This uses the actual upstream browser-capable interface. Native desktop-shell features that require Tauri windows, system tray, OS notifications, or desktop audio capture are not available in a remote browser tab. Use the Studio's in-page import and export controls. The Linux package is marked experimental upstream; Kaggle's container and GPU driver are another compatibility layer, so the first start is a real runtime check, not a guarantee.

The Cloudflare URL and password are temporary and only work while the Kaggle session and processes remain alive. Anyone with the URL still needs the generated password. Do not share the password publicly. Save a Kaggle notebook version/output to preserve songs and model downloads between sessions.

In [ ]:
import os
import re
import sys
import json
import time
import stat
import shutil
import secrets
import hashlib
import tarfile
import subprocess
from pathlib import Path

import requests

STUDIO_VERSION = "5.0.0"
STUDIO_TAG = "v5.0.0"
MANIFEST_NAME = "yue2-studio-assets.json"
WORK = Path("/kaggle/working/yue2_studio_runtime")
WORK.mkdir(parents=True, exist_ok=True)
LOGS = WORK / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

def run(command, check=True, **kwargs):
    return subprocess.run(command, check=check, text=True, **kwargs)

def tail(path, lines=70):
    path = Path(path)
    if not path.exists():
        return "(log not created)"
    return "\n".join(path.read_text(errors="replace").splitlines()[-lines:])

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Stop processes from a previous run in this same notebook kernel.
for old_name in ("CLOUDFLARED_PROCESS", "VITE_PROCESS", "STUDIO_PROCESS"):
    old = globals().get(old_name)
    if old is not None and old.poll() is None:
        old.terminate()
        try:
            old.wait(timeout=12)
        except Exception:
            old.kill()
            old.wait(timeout=5)

# Locate the assets folder whether attached as notebook output or a Kaggle Dataset.
manifest_candidates = []
for root in (Path("/kaggle/input"), Path("/kaggle/working")):
    if root.exists():
        manifest_candidates.extend(root.rglob(MANIFEST_NAME))
if not manifest_candidates:
    raise FileNotFoundError(
        "Cannot find yue2-studio-assets.json. Attach the output of "
        "YuE2_Studio_5_0_0_Assets_Builder.ipynb under Add Input."
    )
manifest_path = manifest_candidates[0]
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest.get("studio_version") != STUDIO_VERSION or manifest.get("studio_tag") != STUDIO_TAG:
    raise RuntimeError(f"Expected YuE2 Studio {STUDIO_TAG}; assets manifest says {manifest.get('studio_tag')}.")

ASSET_DIR = manifest_path.parent
APPIMAGE_NAME = manifest["appimage"]["name"]
source_appimage = ASSET_DIR / APPIMAGE_NAME
source_cloudflared = ASSET_DIR / manifest["cloudflared"]["name"]
if not source_appimage.is_file() or not source_cloudflared.is_file():
    raise FileNotFoundError(f"Builder assets are incomplete in {ASSET_DIR}")

APPIMAGE = WORK / APPIMAGE_NAME
CLOUDFLARED = WORK / "cloudflared-linux-amd64"
shutil.copy2(source_appimage, APPIMAGE)
shutil.copy2(source_cloudflared, CLOUDFLARED)
APPIMAGE.chmod(APPIMAGE.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
CLOUDFLARED.chmod(CLOUDFLARED.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

for path, item in ((APPIMAGE, manifest["appimage"]), (CLOUDFLARED, manifest["cloudflared"])):
    digest = sha256_file(path)
    if digest.lower() != item["sha256"].lower():
        raise RuntimeError(f"SHA-256 mismatch after copying {path.name}: {digest}")
print(f"Verified official Studio {STUDIO_VERSION} AppImage and cloudflared {manifest['cloudflared_version']}.")

# Tauri/WebKit dependencies plus a virtual display for the upstream headless Linux shell.
print("Installing Linux runtime dependencies...")
run(["bash", "-lc",
     "apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends "
     "xvfb xauth dbus-x11 libwebkit2gtk-4.1-0 libayatana-appindicator3-1 "
     "libgtk-3-0 librsvg2-2 libasound2 libgomp1 libvulkan1 ca-certificates "
     "gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad "
     "gstreamer1.0-libav git curl xz-utils >/tmp/yue2-studio-apt.log 2>&1 || "
     "{ tail -80 /tmp/yue2-studio-apt.log; exit 1; }"])

# Vite 8 needs Node >=20.19 or >=22.12. Install a checksum-verified Node 22 LTS only if needed.
def installed_node_ok():
    if not shutil.which("node") or not shutil.which("npm"):
        return False
    value = subprocess.check_output(["node", "--version"], text=True).strip().lstrip("v")
    match = re.match(r"(\d+)\.(\d+)\.(\d+)", value)
    if not match:
        return False
    major, minor, patch = map(int, match.groups())
    return (major == 20 and minor >= 19) or (major == 22 and minor >= 12) or major > 22

if not installed_node_ok():
    node_version = "22.14.0"
    node_name = f"node-v{node_version}-linux-x64.tar.xz"
    base = f"https://nodejs.org/dist/v{node_version}"
    checksum_text = requests.get(f"{base}/SHASUMS256.txt", timeout=60).text
    node_sha = None
    for line in checksum_text.splitlines():
        fields = line.split()
        if len(fields) >= 2 and fields[-1].lstrip("*") == node_name:
            node_sha = fields[0]
            break
    if not node_sha:
        raise RuntimeError(f"Official Node checksum not found for {node_name}.")
    archive = WORK / node_name
    if not archive.exists() or sha256_file(archive) != node_sha:
        with requests.get(f"{base}/{node_name}", stream=True, timeout=(30, 300)) as response:
            response.raise_for_status()
            with open(archive, "wb") as out:
                for chunk in response.iter_content(8 * 1024 * 1024):
                    if chunk:
                        out.write(chunk)
        if sha256_file(archive) != node_sha:
            raise RuntimeError("Node.js archive failed its official SHA-256 check.")
    node_root = WORK / f"node-v{node_version}-linux-x64"
    if not node_root.exists():
        node_root.mkdir(parents=True, exist_ok=True)
        run(["tar", "-xJf", str(archive), "-C", str(WORK), "--strip-components=1"])
    os.environ["PATH"] = str(node_root / "bin") + os.pathsep + os.environ["PATH"]

if not installed_node_ok():
    # The unpacked Node directory may have been used; prepend it explicitly.
    node_bins = sorted(WORK.glob("node-v*/bin"))
    if node_bins:
        os.environ["PATH"] = str(node_bins[-1]) + os.pathsep + os.environ["PATH"]
if not installed_node_ok():
    raise RuntimeError("A supported Node/npm version is not available after setup.")
print("Node:", subprocess.check_output(["node", "--version"], text=True).strip())

# Get the actual upstream source at the same release tag as the official Linux executable.
SOURCE = WORK / "YuE2-Studio"
if not (SOURCE / ".git").exists():
    run([
        "git", "clone", "--depth", "1", "--branch", STUDIO_TAG, "--single-branch",
        "https://github.com/timoncool/YuE2-Studio.git", str(SOURCE)
    ])
else:
    run(["git", "-C", str(SOURCE), "fetch", "--depth", "1", "origin", STUDIO_TAG])
    run(["git", "-C", str(SOURCE), "checkout", "--force", "FETCH_HEAD"])

# Protect the Cloudflare Quick Tunnel with HTTP Basic Auth and disable HMR's external websocket.
# The frontend and API proxy remain the upstream files; this adds only an access gate to the Vite server.
vite_config = SOURCE / "app" / "vite.config.ts"
vite_text = vite_config.read_text(encoding="utf-8")
gate_source = r'''
const jjpStudioGate = {
  name: 'jjp-studio-basic-auth',
  configureServer(server) {
    server.middlewares.use((req, res, next) => {
      const password = process.env.JJP_STUDIO_GATE_PASSWORD;
      if (!password) return next();
      const expected = 'Basic ' + Buffer.from('studio:' + password).toString('base64');
      if (req.headers.authorization !== expected) {
        res.statusCode = 401;
        res.setHeader('WWW-Authenticate', 'Basic realm="JesusJuiceProductions YuE2 Studio"');
        res.end('Studio sign-in required');
        return;
      }
      next();
    });
  }
};

'''
if "jjp-studio-basic-auth" not in vite_text:
    vite_text = vite_text.replace("export default defineConfig", gate_source + "export default defineConfig", 1)
    vite_text = vite_text.replace(
        "    server: {\n      port: 3791,",
        "    server: {\n      hmr: false,\n      allowedHosts: ['.trycloudflare.com'],\n      port: 3791,",
        1,
    )
    vite_text = vite_text.replace(
        "plugins: [react(), tailwindcss()],",
        "plugins: [jjpStudioGate, react(), tailwindcss()],",
        1,
    )
    if "jjpStudioGate" not in vite_text or "allowedHosts" not in vite_text:
        raise RuntimeError("Could not safely add the Cloudflare access gate to the upstream Vite config.")
    vite_config.write_text(vite_text, encoding="utf-8")

print("Installing the upstream Studio frontend dependencies (npm ci)...")
run(["npm", "ci", "--no-audit", "--no-fund"], cwd=SOURCE / "app")

# Keep the Studio library, downloaded weights, settings and songs in Kaggle Output.
STATE = Path("/kaggle/working/.local")
XDG_DATA = STATE / "share"
XDG_CONFIG = STATE / "config"
XDG_CACHE = STATE / "cache"
XDG_RUNTIME = WORK / "xdg-runtime"
for folder in (XDG_DATA, XDG_CONFIG, XDG_CACHE, XDG_RUNTIME):
    folder.mkdir(parents=True, exist_ok=True)
XDG_RUNTIME.chmod(0o700)

studio_env = os.environ.copy()
studio_env.update({
    "APPIMAGE_EXTRACT_AND_RUN": "1",
    "GDK_BACKEND": "x11",
    "WEBKIT_DISABLE_COMPOSITING_MODE": "1",
    "WEBKIT_DISABLE_DMABUF_RENDERER": "1",
    "NO_AT_BRIDGE": "1",
    "XDG_DATA_HOME": str(XDG_DATA),
    "XDG_CONFIG_HOME": str(XDG_CONFIG),
    "XDG_CACHE_HOME": str(XDG_CACHE),
    "XDG_RUNTIME_DIR": str(XDG_RUNTIME),
    "YUE_STUDIO_IDLE_SHUTDOWN_S": "86400",
})

studio_log = LOGS / "studio-headless.log"
print("Starting the official YuE2 Studio backend in headless mode...")
with open(studio_log, "w", encoding="utf-8") as log_file:
    STUDIO_PROCESS = subprocess.Popen(
        [
            "dbus-run-session", "--",
            "xvfb-run", "-a", "-s", "-screen 0 1600x1000x24",
            str(APPIMAGE), "--headless",
        ],
        cwd=WORK,
        env=studio_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

# The real Studio service listens locally on port 8791.
api_url = "http://127.0.0.1:8791/health"
api_ready = False
for attempt in range(90):
    if STUDIO_PROCESS.poll() is not None:
        raise RuntimeError(
            f"YuE2 Studio exited during startup (code {STUDIO_PROCESS.returncode}).\n"
            + tail(studio_log)
        )
    try:
        response = requests.get(api_url, timeout=4)
        if response.status_code < 500:
            api_ready = True
            print("Studio API ready:", response.status_code)
            break
    except requests.RequestException:
        pass
    time.sleep(2)
if not api_ready:
    raise RuntimeError("Studio API did not become ready on port 8791.\n" + tail(studio_log))

# Start the matching v5.0.0 React interface. Its upstream Vite config proxies Studio API routes to localhost:8791.
gate_password = secrets.token_urlsafe(18)
vite_env = studio_env.copy()
vite_env["JJP_STUDIO_GATE_PASSWORD"] = gate_password
vite_log = LOGS / "studio-vite.log"
print("Starting the actual YuE2 Studio browser interface...")
with open(vite_log, "w", encoding="utf-8") as log_file:
    VITE_PROCESS = subprocess.Popen(
        ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", "3791", "--strictPort"],
        cwd=SOURCE / "app",
        env=vite_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

ui_ready = False
for attempt in range(60):
    if VITE_PROCESS.poll() is not None:
        raise RuntimeError(
            f"Vite exited during startup (code {VITE_PROCESS.returncode}).\n"
            + tail(vite_log)
        )
    try:
        response = requests.get("http://127.0.0.1:3791/", auth=("studio", gate_password), timeout=4)
        if response.status_code == 200 and "YuE2" in response.text:
            ui_ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)
if not ui_ready:
    raise RuntimeError("Studio browser UI did not become ready on port 3791.\n" + tail(vite_log))

# Sanity-check the Vite proxy to the real Studio backend before exposing it.
proxy_check = requests.get(
    "http://127.0.0.1:3791/health",
    auth=("studio", gate_password),
    timeout=10,
)
if proxy_check.status_code >= 500:
    raise RuntimeError(f"Vite-to-Studio API proxy check failed: {proxy_check.status_code} {proxy_check.text[:500]}")

# Start Cloudflare Quick Tunnel. The password gate protects every proxied path, including /mcp.
cloudflared_log = LOGS / "cloudflared.log"
with open(cloudflared_log, "w", encoding="utf-8") as log_file:
    CLOUDFLARED_PROCESS = subprocess.Popen(
        [
            str(CLOUDFLARED), "tunnel",
            "--no-autoupdate",
            "--loglevel", "info",
            "--logfile", str(cloudflared_log),
            "--url", "http://127.0.0.1:3791",
        ],
        cwd=WORK,
        env=studio_env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

public_url = None
for attempt in range(90):
    if CLOUDFLARED_PROCESS.poll() is not None:
        raise RuntimeError(
            f"cloudflared exited (code {CLOUDFLARED_PROCESS.returncode}).\n"
            + tail(cloudflared_log)
        )
    log_text = tail(cloudflared_log, 180)
    match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", log_text, re.I)
    if match:
        public_url = match.group(0)
        break
    time.sleep(2)
if not public_url:
    raise RuntimeError("Cloudflare Quick Tunnel did not provide a public URL.\n" + tail(cloudflared_log))

# Verify the public origin itself before printing it.
try:
    edge_check = requests.get(public_url, auth=("studio", gate_password), timeout=25)
    print("Cloudflare edge check:", edge_check.status_code)
except requests.RequestException as exc:
    print("Cloudflare is up; the first browser request will complete the edge check:", str(exc)[:240])

print("\n" + "=" * 72)
print("YUЕ2 STUDIO 5.0.0 IS RUNNING")
print("Studio URL :", public_url)
print("Username   : studio")
print("Password   :", gate_password)
print("Studio data:", XDG_DATA / "yue2-studio")
print("Logs       :", LOGS)
print("=" * 72)
print("Keep this Kaggle session alive while using the Studio. Use Kaggle Save Version to preserve data.")